<a href="https://colab.research.google.com/github/kutayeroglu/biomimetic-training/blob/main/eval_biomim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pickle
import time
import torch

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Define paths
drive_path = '/content/drive/MyDrive/colab_datasets/biomimetic_training'
local_path = '/content/dataset'

In [4]:
# Ensure the local directory exists
if not os.path.exists(local_path):
    print(f"[INFO] Creating local directory: {local_path}")
    os.makedirs(local_path, exist_ok=True)
else:
    print(f"[INFO] Local directory already exists: {local_path}")

# Copy zip from Drive to local VM
zip_file = "style-transfer-preprocessed-512-flat.zip"
drive_zip_path = os.path.join(drive_path, zip_file)

if os.path.exists(drive_zip_path):
    print(f"[INFO] Copying {zip_file} to local VM... (this may take a moment)")
    start_time = time.time()
    !cp "{drive_zip_path}" /content/
    duration = time.time() - start_time
    print(f"[SUCCESS] Copy complete. Time taken: {duration:.2f} seconds.")
else:
    print(f"[ERROR] Could not find {zip_file} at {drive_path}")

# Unzip to the local dataset folder
# [Inference] Checking if the folder is already populated to avoid redundant unzipping
if not os.listdir(local_path):
    print(f"[INFO] Unzipping dataset to {local_path}...")
    start_time = time.time()
    !unzip -q /content/{zip_file} -d {local_path}
    duration = time.time() - start_time
    print(f"[SUCCESS] Extraction complete. Time taken: {duration:.2f} seconds.")
else:
    print(f"[SKIP] {local_path} is not empty. Skipping unzip.")

[INFO] Creating local directory: /content/dataset
[INFO] Copying style-transfer-preprocessed-512-flat.zip to local VM... (this may take a moment)
[SUCCESS] Copy complete. Time taken: 3.63 seconds.
[INFO] Unzipping dataset to /content/dataset...
[SUCCESS] Extraction complete. Time taken: 1.01 seconds.


In [5]:
# Load class indices
with open(os.path.join(drive_path, 'categories16_class_indices.pkl'), 'rb') as f:
    class_indices = pickle.load(f)

# Load the AlexNet checkpoint
checkpoint = torch.load(os.path.join(drive_path, 'standard_checkpoint.pth'), map_location='cpu')

print("Files loaded successfully.")

# Check if it's a dictionary or a direct model object
if isinstance(checkpoint, dict):
    print("Detected format: State Dictionary / Full Checkpoint")
    print("Keys found in checkpoint:", checkpoint.keys())
else:
    print("Detected format: Direct Model Object (not recommended for flexible loading)")

Files loaded successfully.
Detected format: State Dictionary / Full Checkpoint
Keys found in checkpoint: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_acc'])


In [6]:
!git clone https://github.com/kutayeroglu/biomimetic-training.git

Cloning into 'biomimetic-training'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 190 (delta 95), reused 135 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 2.42 MiB | 33.94 MiB/s, done.
Resolving deltas: 100% (95/95), done.


In [7]:
import sys
# Ensure you are in the repo directory
%cd /content/biomimetic-training
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

/content/biomimetic-training


In [8]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.0 MB/s eta 0:00:00


In [9]:
from src.eval.evaluate_bias import run_evaluation

results = run_evaluation(
    model_path=os.path.join(drive_path, 'standard_checkpoint.pth'),
    data_path=os.path.join(local_path, "style-transfer-preprocessed-512-flat"),
    class_indices_path=os.path.join(drive_path, 'categories16_class_indices.pkl'),
    model_file="standard_alexnet",
    result_path="evaluation_results.csv"
)

--- Starting Evaluation on: cuda ---
Configuration:
  Model path: /content/drive/MyDrive/colab_datasets/biomimetic_training/standard_checkpoint.pth
  Data path: /content/dataset/style-transfer-preprocessed-512-flat
  Class indices path: /content/drive/MyDrive/colab_datasets/biomimetic_training/categories16_class_indices.pkl
  Model file identifier: standard_alexnet
  Ranking indices: ['color', 'fft_freq']
  Max ablation: 48
  Top color pixels: 48
  Result path: evaluation_results.csv
  Overwrite: False
--------------------------------------------------------------------------------

[Step 1/7] Loading 16-category class indices...
  Reading from: /content/drive/MyDrive/colab_datasets/biomimetic_training/categories16_class_indices.pkl
  ✓ Loaded 16 categories: ['knife', 'keyboard', 'elephant', 'bicycle', 'airplane', 'clock', 'oven', 'chair', 'bear', 'boat', 'cat', 'bottle', 'truck', 'car', 'bird', 'dog']

[Step 2/7] Loading test dataset...
  Searching for PNG images in: /content/dataset/

/content/biomimetic-training/src/eval/evaluate_bias.py:244: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  res_pd[col_name] = [


    ✓ Predictions complete. Shape: (1280, 1000)
    Counting shape-texture statistics...
    ✓ Statistics complete. Processed: 1200, Skipped: 80
    Sample counts for 'knife': shape=0, other=67, texture=8
    ✓ Saved to column: color_reverse_standard_alexnet_ablation_3

  [103/196 (52.6%)] Ablation 4/48 (color_reverse)
    Zeroed out 4 filters: [36, 24, 13, 11]
    Making predictions on 1280 images...


/content/biomimetic-training/src/eval/evaluate_bias.py:244: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  res_pd[col_name] = [


    ✓ Predictions complete. Shape: (1280, 1000)
    Counting shape-texture statistics...
    ✓ Statistics complete. Processed: 1200, Skipped: 80
    Sample counts for 'knife': shape=0, other=68, texture=7
    ✓ Saved to column: color_reverse_standard_alexnet_ablation_4

  [104/196 (53.1%)] Ablation 5/48 (color_reverse)
    Zeroed out 5 filters: [36, 24, 13, 11, 43]
    Making predictions on 1280 images...


...

/content/biomimetic-training/src/eval/evaluate_bias.py:244: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  res_pd[col_name] = [


    ✓ Predictions complete. Shape: (1280, 1000)
    Counting shape-texture statistics...
    ✓ Statistics complete. Processed: 1200, Skipped: 80
    Sample counts for 'knife': shape=0, other=70, texture=5


/content/biomimetic-training/src/eval/evaluate_bias.py:244: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  res_pd[col_name] = [


    ✓ Saved to column: fft_freq_reverse_standard_alexnet_ablation_47

  [196/196 (100.0%)] Ablation 48/48 (fft_freq_reverse)
    Zeroed out 48 filters: [15, 6, 46, 23, 18, 29, 11, 10, 47, 14, 32, 3, 17, 26, 13, 41, 5, 45, 31, 43, 30, 22, 35, 24, 9, 0, 1, 33, 2, 42, 38, 25, 39, 20, 40, 28, 34, 37, 27, 12, 16, 4, 8, 7, 44, 19, 21, 36]
    Making predictions on 1280 images...
    ✓ Predictions complete. Shape: (1280, 1000)
    Counting shape-texture statistics...
    ✓ Statistics complete. Processed: 1200, Skipped: 80
    Sample counts for 'knife': shape=0, other=70, texture=5
    ✓ Saved to column: fft_freq_reverse_standard_alexnet_ablation_48

Ablation experiments complete!
Total columns in results: 196
Total categories: 16

[Final Step] Saving results...
  Target path: evaluation_results.csv
  Creating new file
  ✓ Results saved successfully to evaluation_results.csv
  Final dataframe shape: (16, 196)

Evaluation Complete!


/content/biomimetic-training/src/eval/evaluate_bias.py:244: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  res_pd[col_name] = [


In [12]:
!ls -lh evaluation_results.csv

-rw-r--r-- 1 root root 49K Dec 30 19:05 evaluation_results.csv


In [13]:
!head -n 5 evaluation_results.csv

,color_standard_alexnet_ablation_0,color_standard_alexnet_ablation_1,color_standard_alexnet_ablation_2,color_standard_alexnet_ablation_3,color_standard_alexnet_ablation_4,color_standard_alexnet_ablation_5,color_standard_alexnet_ablation_6,color_standard_alexnet_ablation_7,color_standard_alexnet_ablation_8,color_standard_alexnet_ablation_9,color_standard_alexnet_ablation_10,color_standard_alexnet_ablation_11,color_standard_alexnet_ablation_12,color_standard_alexnet_ablation_13,color_standard_alexnet_ablation_14,color_standard_alexnet_ablation_15,color_standard_alexnet_ablation_16,color_standard_alexnet_ablation_17,color_standard_alexnet_ablation_18,color_standard_alexnet_ablation_19,color_standard_alexnet_ablation_20,color_standard_alexnet_ablation_21,color_standard_alexnet_ablation_22,color_standard_alexnet_ablation_23,color_standard_alexnet_ablation_24,color_standard_alexnet_ablation_25,color_standard_alexnet_ablation_26,color_standard_alexnet_ablation_27,color_standard_alexnet_ablati

In [14]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the source and destination paths
source_file = 'evaluation_results.csv'
dest_folder = '/content/drive/MyDrive/colab_results/biomimetic_training/'
dest_path = os.path.join(dest_folder, source_file)

# 3. Create the directory if it doesn't exist
if not os.path.exists(dest_folder):
    os.makedirs(dest_folder)
    print(f"Created directory: {dest_folder}")

# 4. Copy the file
!cp {source_file} {dest_path}

# 5. Verify the file exists in Drive
if os.path.exists(dest_path):
    print(f"Successfully saved to: {dest_path}")
else:
    print("Error: File was not saved.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Created directory: /content/drive/MyDrive/colab_results/biomimetic_training/
Successfully saved to: /content/drive/MyDrive/colab_results/biomimetic_training/evaluation_results.csv
